# PROGRES 2023 - Mini-Projet 2
# API Web

Fabien Mathieu - fabien.mathieu@lip6.fr

Sébastien Tixeuil - Sebastien.Tixeuil@lip6.fr

The purpose of this mini-project is to work with the *Internet Movie DataBase* (IMDB) and a Python Web framework. It will involve:

- Retrieve and manipulate datasets
- Build an API to perform various tasks on the data
- Build a website that will use the API above

# The IMDB dataset

[IMDB](https://www.imdb.com) allows to retrieve a part of its dataset for any non-commercial purpose. The available data and the formatting convention is described here: https://developer.imdb.com/non-commercial-datasets/

We are especially interested in the data from the following files:
- https://datasets.imdbws.com/title.principals.tsv.gz
- https://datasets.imdbws.com/name.basics.tsv.gz
- https://datasets.imdbws.com/title.basics.tsv.gz

**Important notes**:
- If you see *Your answer here*, that means something is expected from you.
- To help you, the start and/or the end of a possible solution is sometimes given.
- The content of IMDB is refreshed regularly. That means that some of the results you will compute, like the number of movies, will vary with time. This should not surprise you.

## Exercise 1: Download

Write a `download_imdb` function inspired by the `download` function seen in course, with the following modifications:
- `download_imdb` will have one single argument, the name of the file to retrieve. Server location of the file is assumed to be https://datasets.imdbws.com/
- If the file already exists, print a message telling that it exists and do nothing. You can use the `pathlib` module for that.
 The data files are quite big, so you will specify a directory `data_dir` where the data files will be stored/read.

Your answer here.

In [1]:
from pathlib import Path
from requests import Session

base_url = "https://datasets.imdbws.com/"
data_dir = Path.home() / "Downloads"


def download_imdb(file):
    target_path = data_dir / file

    if target_path.exists():
        print(f"{target_path} already exists")
        return

    url = base_url + file
    s = Session()
    r = s.get(url, stream=True)
    with open(target_path, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

In [2]:
files = ['title.principals.tsv.gz', 'name.basics.tsv.gz', 'title.basics.tsv.gz']
for file in files:
    download_imdb(file)

## Exercise 2: Explore

- What is the size of the different files you retrieved? You can use Python or a file explorer, as you prefer.

Your answer here.

In [3]:
p_name = Path(data_dir/"name.basics.tsv.gz")
p_principals = Path(data_dir/"title.principals.tsv.gz")
p_basics = Path(data_dir/"title.basics.tsv.gz")

size_mb_name = p_name.stat().st_size / (1024 * 1024)
size_mb_principals = p_principals.stat().st_size / (1024 * 1024)
size_mb_basics = p_basics.stat().st_size / (1024 * 1024)

print(f"The size of name.basics.tsv.gz is {size_mb_name} MB")
print(f"The size of title.principals.tsv.gz is {size_mb_principals} MB")
print(f"The size of title.basics.tsv.gz is {size_mb_basics} MB")

The size of name.basics.tsv.gz is 282.5453567504883 MB
The size of title.principals.tsv.gz is 711.7204542160034 MB
The size of title.basics.tsv.gz is 205.500394821167 MB


As explained in https://developer.imdb.com/non-commercial-datasets/:
- the data is stored as `tsv`, which means each text line represents a row.
- A [gzip compression](https://docs.python.org/3/library/gzip.html) is used to reduce the size of the data on the hard drive.

Large compressed files should not be uncompressed on your hard drive or fully loaded in memory.

The Python [gzip module](https://docs.python.org/3/library/gzip.html) is designed so you can open a compressed file as if it was already uncompressed. For example, the following code reads 666 lines from `title.basics` and print the last line read.

In [4]:
import gzip
with gzip.open(data_dir / 'title.basics.tsv.gz', 'rt', encoding='utf8') as f:
    for _ in range(666):
        l = f.readline()
print(l)

tt0000671	short	Desdemona	Desdemona	0	1908	\N	\N	Drama,Short



- Write a function that read the 4 first lines of a compressed tsv file. Each line read should be converted into a list of elements and printed.

Your answer here.

In [5]:
def explore(name):
    with gzip.open(data_dir / name, 'rt', encoding='utf8') as f:
        for _ in range(4):
            l = f.readline()
            print(l.split("\t"))

In [6]:
for file in files:
    print("")
    print(f"First lines of {file}:")
    explore(file)


First lines of title.principals.tsv.gz:
['tconst', 'ordering', 'nconst', 'category', 'job', 'characters\n']
['tt0000001', '1', 'nm1588970', 'self', '\\N', '["Self"]\n']
['tt0000001', '2', 'nm0005690', 'director', '\\N', '\\N\n']
['tt0000001', '3', 'nm0005690', 'producer', 'producer', '\\N\n']

First lines of name.basics.tsv.gz:
['nconst', 'primaryName', 'birthYear', 'deathYear', 'primaryProfession', 'knownForTitles\n']
['nm0000001', 'Fred Astaire', '1899', '1987', 'actor,miscellaneous,producer', 'tt0072308,tt0050419,tt0027125,tt0025164\n']
['nm0000002', 'Lauren Bacall', '1924', '2014', 'actress,miscellaneous,soundtrack', 'tt0037382,tt0075213,tt0038355,tt0117057\n']
['nm0000003', 'Brigitte Bardot', '1934', '\\N', 'actress,music_department,producer', 'tt0057345,tt0049189,tt0056404,tt0054452\n']

First lines of title.basics.tsv.gz:
['tconst', 'titleType', 'primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'endYear', 'runtimeMinutes', 'genres\n']
['tt0000001', 'short', 'Carmencita',

- How many movie entries are present in the retrieved database?
- How many people entries?

Your answer here.

In [7]:
import gzip

x = set()

with gzip.open(data_dir / 'title.basics.tsv.gz', 'rt', encoding='utf8') as f:
    header = f.readline().split('\t')
    tconst_idx = header.index('tconst')
    type_idx = header.index('titleType')

    for line in f:
        cols = line.split('\t')
        if cols[type_idx] == 'movie':
            x.add(cols[tconst_idx])

print(len(x))

735164


In [8]:
res = 0
with gzip.open(data_dir / 'title.basics.tsv.gz', 'rt', encoding='utf8') as f:

  while(True):
      l = f.readline()
      if(l==""):
        break
      x =l.split("\t")
      if(x[1]=="movie"):
         res=res+1
print(res)

735164


In [9]:
res = 0
with gzip.open(data_dir / 'name.basics.tsv.gz', 'rt', encoding='utf8') as f:

  while(True):
      l = f.readline()
      if(l==""):
        break
      res=res+1
print(res)

14968593


## Exercise 3: Extract

We want to study the relations between actors and movies. In particular, we focus on:
- Actual movies (e.g. not TV shows or short movies), where the movie year is known and at least one actor/actress is credited.
- Actors that are credited in at least one actual movie.

To start with, build a [Python set](https://docs.python.org/3/tutorial/datastructures.html#sets) that contains all movie ids (`tconst`) such that:
- The type of movie (`titleType`) is `movie`;
- The year (`startYear`) exists, i.e. is an integer.

How many movies have you referenced in the set?

Your answer here.

so in here we notice the same logic, run line by line  a gzip, and if conditions meets, add.

so here if its a movie and if a integer and present!

After consulting the format in the past exercises, I knew the indexes of each value, and noticed that some integers have \\N, which logically should mean it is the value that indicates the absence of a number

In [10]:
true_movies = set()
with gzip.open(data_dir / 'title.basics.tsv.gz', 'rt', encoding='utf8') as f:

  while(True):
      l = f.readline()
      if(l==""):
        break
      x =l.split("\t")
      if(x[1]=="movie"):
        if(x[5]!="" and x[5]!='\\N'):
          true_movies.add(x[0])

In [11]:
len(true_movies)

626404

Now we want to build two lists, `movies` and `actors`:

- Each element of `movies` should represent a movie, each element of `actors` an actor or actress;
- A movie is represented by a list of three elements:
  - The original name of the movie (`str`),
  - The principal actors of the movie, stored as a list whose elements are integers that represent the index (position) of the actors in the list `actors`,
  - The movie year, `startYear` (`int`);
- An actor/actress is represented by a list of two elements:
  - The name of the person (`str`),
  - The movies the person acted in, stored as a list whose elements are integers that represent the index (position) of the movies in the list `movies`.
  

Build these two lists.

A possible way to do this (this is a suggestion, not an order):
- Initiate `movies` and `actors` as empty lists;
- Create two auxiliary dictionary that will associate to each movie id (`tconst`) and person id (`nconst`) their position in the list;
- Read the file `title.principals.tsv.gz` line by line:
  - Ignore any line where the movie is not in the set `true_movies` or the `category` of the relation is not `actor` or `actress`,
  - If the movie id `tconst` is not in the movie auxiliary index, append an empty movie to `movies` (`["", [], 0]`) and update the movie auxiliary index with an entry for `tconst`,
  - If the actor id `nconst` is not in the actor auxiliary index, append an empty actor to `actors` (`["", []]`) and update the actor auxiliary index with an entry for `nconst`,
  - Append the movie index (not `tconst`!) to the movies of the corresponding actor in `actors`,
  - Append the actor index (not `nconst`!) to the actors of the corresponding movie in `movies`;
- There can be a few undesired duplicates, e.g. some actors can have multiple entries for the same movies. For each actor, remove possible duplicates in the list of movies, and for each movie, remove possible duplicates in the list of actors;
- Using `title.basics.tsv.gz` and your movie auxiliary index, populate each movie in `movies` with its correct name (`str`) and year (`int`);
- Using `name.basics.tsv.gz` and your actor auxiliary index, populate each actor in `movies` with her correct name.

Your answer here.

In [12]:
movie_id_to_index = dict()
movies = []


actor_id_to_index = dict()
actors = []

In [13]:
with gzip.open(data_dir / "title.principals.tsv.gz", "rt", encoding="utf8") as f:
    while(True):
        l = f.readline()
        if l == "":
            break
        x = l.strip().split("\t")
        tconst = x[0]
        nconst = x[2]
        category = x[3]

        if tconst not in true_movies:
            continue
        if category !="actor" and category!= "actress":
            continue

        # moviee
        if tconst not in movie_id_to_index:
            movie_id_to_index[tconst] = len(movies)
            movies.append(["", [], 0])

        # actor
        if nconst not in actor_id_to_index:
            actor_id_to_index[nconst] = len(actors)
            actors.append(["", []])

        m_i = movie_id_to_index[tconst]
        a_i = actor_id_to_index[nconst]

        movies[m_i][1].append(a_i) # ajoute indx actor to oviue
        actors[a_i][1].append(m_i)  # ajoute index movie to actor

In [14]:
for movie in movies:
    actor_indices = movie[1]
    unique_actors = set(actor_indices)
    movie[1] = list(unique_actors)

for actor in actors:
    movie_indices = actor[1]
    unique_movies = set(movie_indices)
    actor[1] = list(unique_movies)

And now I populated the result with the needed data, again with the help of the suggestion which was very helpful

In [15]:
with gzip.open(data_dir / "title.basics.tsv.gz", "rt", encoding="utf8") as f:
     while(True):
        l = f.readline()
        if l == "":
            break
        x = l.strip().split("\t")
        tconst = x[0]

        if tconst in movie_id_to_index:
            idx = movie_id_to_index[tconst]
            movies[idx][0] = x[3]
            movies[idx][2] = int(x[5])

In [16]:
with gzip.open(data_dir / "name.basics.tsv.gz", "rt", encoding="utf8") as f:
    while(True):
        l = f.readline()
        if l == "":
            break
        x = l.strip().split("\t")
        nconst = x[0]

        if nconst in actor_id_to_index:
            idx = actor_id_to_index[nconst]
            actors[idx][0] = x[1]

In [17]:
print(movies[0])
print(actors[0])

['Miss Jerry', [0, 1, 2], 1894]
['Blanche Bayliss', [0]]


Manually check that your files are correct. For example, try to get the name and year of the movies Michel Blanc played in, or the actors of the first Harry Potter movie.

Your answer here (if everything went well, you just need to execute the two cells below).

In [18]:
', '.join([f"{movies[i][0]} ({movies[i][2]})" for i in [a for a in actors if a[0]=='Michel Blanc'][0][1]])

"Demi-soeur (2013), Les Tuche 4 (2021), La fille du RER (2009), La meilleure façon de marcher (1976), Et soudain, tout le monde me manque (2011), Raid dingue (2016), Docteur? (2019), Les grands ducs (1996), The Favour, the Watch and the Very Big Fish (1991), The Hundred-Foot Journey (2014), Je vous trouve très beau (2005), Papy fait de la résistance (1983), Rien ne va plus (1979), Nemo (1984), Cause toujours... tu m'intéresses! (1979), Marie-Line et son juge (2023), Circulez y a rien à voir! (1983), Uranus (1990), Embrassez qui vous voudrez (2002), Le cheval d'orgueil (1980), Musée haut, musée bas (2008), Nos 18 ans (2008), Les témoins (2007), Le routard (2025), Ma femme s'appelle reviens (1982), Prospero's Books (1991), Les souvenirs (2014), Chambre à part (1989), Toxic Affair (1993), Madame Edouard (2004), Le beaujolais nouveau est arrivé (1978), Les nouvelles aventures d'Aladin (2015), Voyez comme on danse (2018), Vous n'aurez pas l'Alsace et la Lorraine (1977), Les cadors (2022), V

In [19]:
', '.join([actors[i][0] for i in [m for m in movies if m[0].startswith('Harry Potter')][0][1]])

'Daniel Radcliffe, Robbie Coltrane, Richard Harris, Fiona Shaw, Harry Melling, Richard Griffiths, Rupert Grint, Emma Watson, Saunders Triplets, Maggie Smith'

When you have successfully reached this point of the project, you can save the two lists `movies` and `actors` as compressed json files using the code below:

In [20]:
import gzip
import json

with gzip.open(data_dir / 'movies.json.gz', 'wt', encoding='utf8') as f:
    json.dump(movies, f)
with gzip.open(data_dir / 'actors.json.gz', 'wt', encoding='utf8') as f:
    json.dump(actors, f)

After your files have been saved, you do not need to re-execute all of the above each time your restart your notebook. Instead, you just need to reload `movies` and `actors` using the code below:

In [21]:
import gzip
import json

with gzip.open(data_dir / 'movies.json.gz', 'rt', encoding='utf8') as f:
    movies = json.load(f)
with gzip.open(data_dir / 'actors.json.gz', 'rt', encoding='utf8') as f:
    actors = json.load(f)

**Important remark:** in what follows, you will have to build functions that use the two lists a lot. You should NOT reload the lists each time you call a function. Instead, ensure that the two lists are loaded in memory and use them directly.

## Exercise 4: Explore again (now on the curated dataset)

- How many actors do you have in the new dataset? How many movies?
- In average, in how many movies played an actor?
- In average, how many actors play in a movie?
- What is the name of the actor that played in the most movies? How many movies did he feature in?
- What is the oldest movie in the DB?

Your answer here.

In [22]:
len(actors)

1264225

In [23]:
len(movies)

492714

In [24]:
c_movies = 0
for actor in actors:
    c_movies = c_movies + len(actor[1])
print(c_movies / len(actors))

3.0020154640194585


In [25]:
c_actors = 0
for movie in movies:
    c_actors = c_actors + len(movie[1])

print(c_actors / len(movies))

7.702689592745487


In [26]:
actor_max = 0
for actor in actors:
    if len(actor[1]) > actor_max:
        actor_max = len(actor[1])
        actor_name = actor[0]

print(actor_name)
print(actor_max)


Brahmanandam
1124


In [27]:
oldes_movie = 300000
for movie in movies:
    if(movie[2] < oldes_movie):
        oldes_movie = movie[2]
        movie_name = movie[0]

print(movie_name)
print(oldes_movie)

Miss Jerry
1894


## Exercise 5: Prepare some functions

Write the following functions
- `search_movie(name: str) -> list`: return a list of movies whose name contains `name` (ignoring case). Each movie is described as a dictionary with keys `name`, `year`, and `index` (its position in `movies`)
- `get_movie(i: int) -> dict`: returns the a json of the movie at position `i`, with following keys:
  - `name` (`str`)
  - `year` (`int`)
  - `actors` (list of dictionaries with keys `name` and `index`)
- `search_actor(name: str) -> list`: return a list of actors whose name contains `name` (ignoring case). Each actor is described as a dictionary with keys `name` and `index` (its position in `actor`)
- `get_actor(i: int) -> dict`: returns the a json of the actor at position `i`, with following keys:
  - `name` (`str`)
  - `movies` (list of dictionaries with keys `name`, `year`, and `index`)

Your answer here.

In [28]:
def search_movie(name):
    ##Renvoie la liste des films contenant le nom
    name_lower = name.lower()
    res = []
    for i in range(len(movies)):
        movie = movies[i]
        if movie[0] != "" and name_lower in movie[0].lower():
            res.append({"name": movie[0],"year": movie[2],"index": i})
    return res

In [29]:
def get_movie(i):
    #Renvoie le dictionnaire détaillé du film à l'index i
    movie = movies[i]
    name = movie[0]
    actors_indexes = movie[1]
    year = movie[2]
    actors_list = []

    for j in actors_indexes:
        actor_name = actors[j][0]
        actors_list.append({"name": actor_name,"index": j})

    return {"name": name,"year": year,"actors": actors_list}

In [30]:
def search_actor(name):
    #Renvoie une liste des acteurs dont le nom contient la chaîne donnée
    results = []
    name_lower = name.lower()

    for i in range(len(actors)):
        actor = actors[i]
        actor_name = actor[0]
        if name_lower in actor_name.lower():
            results.append({"name": actor_name,"index": i})

    return results

In [31]:
def get_actor(i):
    #Renvoie les détails de l'acteur à l'index i
    actor = actors[i]
    actor_name = actor[0]
    movies_indexes = actor[1]
    movies_list = []

    for j in movies_indexes:
        movie = movies[j]
        movie_name = movie[0]
        movie_year = movie[2]

        movies_list.append({"name": movie_name,"year": movie_year,"index": j})

    return {"name": actor_name,"movies": movies_list}

In [32]:
bronzés = search_movie('bronzés')
bronzés

[{'name': 'Les bronzés', 'year': 1978, 'index': 54000},
 {'name': 'Les bronzés font du ski', 'year': 1979, 'index': 55034},
 {'name': 'Les bronzés 3: amis pour la vie', 'year': 2006, 'index': 180716},
 {'name': "Les P'tits Bronzés au Pyrénéen", 'year': 2013, 'index': 467484}]

In [33]:
search_movie('gendarme')

[{'name': 'Le gendarme de Saint-Tropez', 'year': 1964, 'index': 41008},
 {'name': 'Le gendarme à New York', 'year': 1965, 'index': 42676},
 {'name': 'Le gendarme se marie', 'year': 1968, 'index': 44459},
 {'name': 'Le gendarme en balade', 'year': 1970, 'index': 46407},
 {'name': 'Le gendarme et les extra-terrestres', 'year': 1979, 'index': 55248},
 {'name': 'Le gendarme et les gendarmettes', 'year': 1982, 'index': 58345},
 {'name': 'Le gendarme de Champignol', 'year': 1959, 'index': 92831},
 {'name': 'El gendarme desconocido', 'year': 1941, 'index': 94279},
 {'name': 'El gendarme de la esquina', 'year': 1951, 'index': 116188},
 {'name': 'Sacrés gendarmes', 'year': 1980, 'index': 120030},
 {'name': "Hainburg - Je t'aime, gendarme", 'year': 2001, 'index': 145906},
 {'name': 'Le gendarme de Abobo', 'year': 2019, 'index': 320671},
 {'name': 'Le retour du gendarme de Abobo', 'year': 2025, 'index': 399616}]

In [34]:
get_movie(search_movie('Ils sont fous')[0]['index'])

{'name': 'Ils sont fous ces sorciers',
 'year': 1978,
 'actors': [{'name': 'Catherine Lachens', 'index': 99392},
  {'name': 'Renée Saint-Cyr', 'index': 23811},
  {'name': 'Michel Peyrelon', 'index': 85253},
  {'name': 'Jean-Jacques Moreau', 'index': 96009},
  {'name': 'Jean Lefebvre', 'index': 25388},
  {'name': 'Maitena Galli', 'index': 81810},
  {'name': 'Julien Guiomar', 'index': 72020},
  {'name': 'Henri Guybet', 'index': 84598},
  {'name': 'Daniel Ceccaldi', 'index': 49815},
  {'name': 'Dominique Vallée', 'index': 244191}]}

In [35]:
get_movie(bronzés[0]['index'])

{'name': 'Les bronzés',
 'year': 1978,
 'actors': [{'name': 'Michel Creton', 'index': 72838},
  {'name': 'Marie-Anne Chazel', 'index': 103879},
  {'name': 'Bruno Moynot', 'index': 103880},
  {'name': 'Thierry Lhermitte', 'index': 103881},
  {'name': 'Michel Blanc', 'index': 99339},
  {'name': 'Gérard Jugnot', 'index': 98987},
  {'name': 'Josiane Balasko', 'index': 101298},
  {'name': 'Dominique Lavanant', 'index': 103318},
  {'name': 'Martin Lamotte', 'index': 103319},
  {'name': 'Luis Rego', 'index': 84378}]}

In [36]:
harry = search_actor('Daniel Radcliffe')
harry

[{'name': 'Daniel Radcliffe', 'index': 278113}]

In [37]:
get_actor(harry[0]['index'])

{'name': 'Daniel Radcliffe',
 'movies': [{'name': 'Escape from Pretoria', 'year': 2020, 'index': 443786},
  {'name': 'Harry Potter and the Deathly Hallows: Part 2',
   'year': 2011,
   'index': 227084},
  {'name': 'Imperium', 'year': 2016, 'index': 428049},
  {'name': 'Horns', 'year': 2013, 'index': 268701},
  {'name': 'Merrily We Roll Along', 'year': 2025, 'index': 376870},
  {'name': 'Harry Potter and the Prisoner of Azkaban',
   'year': 2004,
   'index': 145912},
  {'name': 'Victor Frankenstein', 'year': 2015, 'index': 299178},
  {'name': 'Swiss Army Man', 'year': 2016, 'index': 416816},
  {'name': 'Playmobil: The Movie', 'year': 2019, 'index': 419386},
  {'name': 'Harry Potter and the Half-Blood Prince',
   'year': 2009,
   'index': 175170},
  {'name': 'Weird: The Al Yankovic Story', 'year': 2022, 'index': 284484},
  {'name': 'The F Word', 'year': 2013, 'index': 263624},
  {'name': 'The Woman in Black', 'year': 2012, 'index': 276680},
  {'name': 'Beast of Burden', 'year': 2018, 'in

Write a function `movie_path(origin: int, destination: int) -> distance: int, path: list` that computes the collaboration distance between two actors. That distance is the length of the shortest path `(origin, act1, act2, ..., actX, destination)`, where `origin` and `act` played in the same movie, `act1` and `act2` played in the same movie, ... and
`actX` and `destination` played in the same movie.  In addition to the distance, the response should include one shortest path between the two actors, as a list of the form `["origin_name", "movie1_name", "act1_name", "movie2_name", ..., "destination_name"]`, where `movie1` is a movie that featured `origin` and `act1`, and so on...

In particular:
- One actor is by convention at distance 0 from herself. The return path should be `["origin_name"]` then;
- Two distinct actors that play in the same movie are at distance 1;
- If there is no connection between two actors, the function should return `-1, []` by convention.

**Important remarks**: `movie_path` is tricky. You need to try to implement it but you are allowed to fail. If you are stuck for too long, please explain what you did/try and what blocked you in your opinion. Then move on.

Your answer here.

In [38]:
from collections import deque

def movie_path(origin, destination):
    """
    Calcule le chemin le plus court entre deux acteurs (BFS)
    Renvoie la distance et la liste des étapes
    """
    # Cas trivial
    if origin == destination:
        return 0, [actors[origin][0]]

    # visited[actor] = (acteur_précédent, film_lien)
    visited = {origin: (None, None)}
    queue = deque([origin])

    while queue:
        current_actor = queue.popleft()

        # Tous les films de l'acteur actuel
        for movie_id in actors[current_actor][1]:

            # tous les co-actors dans ce film
            for next_actor in movies[movie_id][1]:

                if next_actor in visited:
                    continue

                # On note comment on est arrivé là
                visited[next_actor] = (current_actor, movie_id)

                # Trouvé
                if next_actor == destination:
                    return reconstruct_path(visited, destination)

                queue.append(next_actor)

    # Pas de connexion trouvée
    return -1, []

def reconstruct_path(visited, destination):
    """Reconstruit le chemin complet en remontant le dictionnaire visited"""

    path = []
    distance = 0
    current = destination

    # On remonte jusqu'à l'origine (où previous_actor sera None)
    while current is not None:
        actor_name = actors[current][0]
        path.append(actor_name)

        previous_actor, movie_id = visited[current]

        if movie_id is not None:
            movie_title = movies[movie_id][0]
            path.append(movie_title)
            distance += 1

        current = previous_actor

    # Le chemin a été construit à l'envers (Destination -> Origine), on l'inverse
    path.reverse()
    return distance, path


In [39]:
jean = search_actor('jean dujardin')
jean_index = jean[0]['index']
jean

[{'name': 'Jean Dujardin', 'index': 330558}]

In [40]:
jack = search_actor('kiefer sutherland')
jack_index = jack[0]['index']
jack

[{'name': 'Kiefer Sutherland', 'index': 123950}]

In [41]:
kevin = search_actor('kevin bacon')
kevin_index = kevin[0]['index']
kevin

[{'name': 'Kevin Bacon', 'index': 105473},
 {'name': 'Kevin Bacon', 'index': 1210023}]

In [42]:
cruchot = search_actor('louis de funès')
cruchot_index = cruchot[0]['index']
cruchot

[{'name': 'Louis de Funès', 'index': 38583}]

In [43]:
movie_path(kevin_index, kevin_index)

(0, ['Kevin Bacon'])

In [44]:
movie_path(kevin_index, jean_index)

(2,
 ['Kevin Bacon',
  'Wild Things',
  'Bill Murray',
  'The Monuments Men',
  'Jean Dujardin'])

In [45]:
movie_path(cruchot_index, jack_index)

(3,
 ['Louis de Funès',
  'Fantômas',
  'Andrée Tainsy',
  'Sous le sable',
  'Charlotte Rampling',
  'Melancholia',
  'Kiefer Sutherland'])

In [46]:
def get_costars(actor_id):
    """
    Renvoie une liste contenant les costars d'un acteur
    """
    actor_movies = get_actor(actor_id)['movies']
    unique_costars = {}

    for movie in actor_movies:
        costars = get_movie(movie['index'])['actors']

        for costar in costars:
            if costar['index'] != actor_id:
                unique_costars[costar['index']] = costar

    return list(unique_costars.values())

get_costars(1)

[{'name': 'Blanche Bayliss', 'index': 0},
 {'name': 'Chauncey Depew', 'index': 2},
 {'name': 'Robert Clugston', 'index': 5130},
 {'name': 'John W. Boyle', 'index': 6032},
 {'name': 'Suzanne Willa', 'index': 4690},
 {'name': 'Mollie King', 'index': 4630},
 {'name': 'Richard Taber', 'index': 2783},
 {'name': 'Dorothy Revier', 'index': 11425},
 {'name': 'Noah Beery', 'index': 5804},
 {'name': 'Louis King', 'index': 8718},
 {'name': 'Douglas Fairbanks Jr.', 'index': 4656},
 {'name': 'Ivan F. Simpson', 'index': 3248},
 {'name': 'William Orlamond', 'index': 8665},
 {'name': 'Robert Edeson', 'index': 1338},
 {'name': 'Henry Kolker', 'index': 2844},
 {'name': 'Anders Randolf', 'index': 3165},
 {'name': 'Josephine Lovett', 'index': 5538},
 {'name': 'William Dangman', 'index': 3504},
 {'name': 'Frank Currier', 'index': 275},
 {'name': 'Wilfred Lytell', 'index': 4885},
 {'name': 'Lucille Lee Stewart', 'index': 4316},
 {'name': 'Marguerite Snow', 'index': 1867},
 {'name': 'Norman Thorpe', 'index':

## Exercise 6. Provide a Web API

Using Python and Flask, build a web server that implements the following routes:
- `/movies/{id}` : where `id` is the index of a movie, returns the corresponding movie as a json (cf `get_movie`).
- `/movies` : returns by default the first 100 movies. The value 100 can be modified by sending a URL parameter `limit`.
- `/actors/{id}` : where `id` is the index of an author, returns the json of the actor (cf `get_actor`).
- `/actors` : returns by default the first 100 actors. The value 100 can be modified by sending a URL parameter `limit`.
- `/actors/{id}/costars` : returns the co-stars of one actor (actors that play in a same movie).
- `/search/actors/{searchString}` : where `searchString` is a string to lookup one actor. This route should return the actors whose name contains `searchString` (for example, `/search/actors/w` returns the actors whose name contains `w` or `W`).
- `/search/movies/{searchString}`: where `searchString` is a string, returns the list of movies whose title contains `searchString`. The route should accept a URL parameter `filter` formatted like `key1:value1,key2:value2,...`  to restrain the search to the publications where key `keyi` contains `valuei`. For example, `/search/movies/gendarme?filter=year:1964`
should return the list of movies where the title contains `gendarme` published in 1964.
- `/actors/{id_origin}/distance/{id_destination}` : where `id_origin`
and `id_destination` are two actor indices, returns the collaboration distance between the two actors. In addition to the distance, the response should include one shortest path between the two actors, e.g. the json you return should be a list of two elements, one integer and one list.

The developed API should have the following characteristics:

- All errors should have the same format.
- In absence of error, the API should always return a `json`.
- Each route must be documented with the return format, possible errors, and an explanation of parameters.
- Each route that returns a list should return a maximum of 100 elements and should accept URL parameters `start` and `limit` to display `limit` elements starting from the `start`-th element. For example: `/actors` should return the first 100 authors, `/actors?start=100` displays the next 100, and `/actors?start=200&limit=2` displays the next 2 elements.
- For each route that returns a list, the returned elements should be sortable based on a given field using a URL parameter `order`. For example: `/movies?order=year` displays the first 100 movies sorted by year.

Your answer here.

# Remarques

## Différence d'url pour l’API et le site graphique
Pour faire la différence entre l’API et le site graphique, nous avons suivi cette convention :<br/>
Si le lien contient `/api`, il s’agit de l’API.<br/>
Sinon, il s’agit de l’interface graphique.<br/>
Exemples :<br/>
`http://127.0.0.1:5000/api/actors/{id_origin}/distance/{id_destination}` : fait appel à l’API<br/>
`http://127.0.0.1:5000/actors/{id_origin}/distance/{id_destination}` : fait appel au site graphique<br/>

## Les URLS valides
base_url = http://127.0.0.1:5000

### API

* `/api/movies` accepte les paramètres `start`, `limit` et `order` (index, name, year)
* `/api/movies/<id>` renvoie les détails d'un film spécifique
* `/api/actors` accepte les paramètres `start`, `limit` et `order` (index, name)
* `/api/actors/<id>` renvoie les détails d'un acteur spécifique
* `/api/actors/<id>/costars` accepte les paramètres `start`, `limit` et `order` (index, name)
* `/api/search/movies/<searchString>` accepte les paramètres `start`, `limit` et `filter` (year)
* `/api/search/actors/<searchString>` accepte les paramètres `start` et `limit`
* `/api/actors/<id_orig>/distance/<id_dest>` renvoie le chemin et la distance entre deux acteurs

### Site Web
* `/movies` accepte les paramètres `start`, `limit` et `order` (index, name, year)
* `/movies/<id>` affiche la page de profil d'un film
* `/actors` accepte les paramètres `start`, `limit` et `order` (index, name)
* `/actors/<id>` affiche la page de profil d'un acteur
* `/search/movies/<searchString>` accepte les paramètres `start`, `limit` et `filter` (year)
* `/search/actors/<searchString>` accepte les paramètres `start` et `limit`
* `/distance` accepte les paramètres `name1` et `name2` (pour la recherche par nom)
* `/actors/<id_orig>/distance/<id_dest>` affiche le résultat graphique de la distance


## Import des fonctions
Pour importer les fonctions depuis le notebook vers nos fichiers Python, nous avons vu plusieurs solutions consistant à utiliser différentes bibliothèques. Cependant, nous avons trouvé que la solution la plus simple consiste à écrire le code dans un fichier `shared_code.py` et à effectuer les imports depuis celui-ci

## Gestion des erreurs


Gérer les erreurs a été difficile. On a essayé plusieurs méthodes (comme `abort` ou `try/except`) mais cela demandait d'écrire trop de code et de réfléchir à chaque cas

Finalement, on a utilisé une solution trouvée sur le site de Flask https://flask.palletsprojects.com/en/stable/errorhandling/#generic-exception-handlers les gestionnaires d'exceptions génériques. Cela nous permet de capturer les erreurs automatiquement

1. **Dans l'API :** On capture les erreurs comme `ValueError` ou `IndexError` pour renvoyer directement du JSON.
2. **Dans le Site :** Quand on appelle l'API, on vérifie si elle répond une erreur. Si oui, on affiche le message avec `abort`.
3. **Dans `run.py` (Global) :** On a un gestionnaire unique qui attrape toutes les autres erreurs. Il décide tout seul s'il doit renvoyer du JSON ou la page `error.html`.

Un problème qu'on a rencontré est que la fonction abort déclenche aussi une erreur qui était capturée par erreur par notre gestionnaire global (@app.errorhandler(Exception)). L'IA nous a aidés à résoudre ça pour bien différencier une erreur volontaire (comme une 404) d'un vrai crash

On ne sait pas si c'est la méthode parfaite, mais c'était la plus pratique pour nous. Enfin, on a décidé d'afficher les détails des erreurs (sans les cacher pour la sécurité) car c'est plus simple pour nous/vous aider à corriger le projet


In [47]:
%%writefile shared_code.py

from collections import deque

# Listes partagées remplies par run.py
movies = []
actors = []

# Fonctions de l'exercice précédent


# Movies
def search_movie(name):
    """Renvoie la liste des films contenant le nom"""
    name_lower = name.lower()
    res = []
    for i in range(len(movies)):
        movie = movies[i]
        if movie[0] != "" and name_lower in movie[0].lower():
            res.append({"name": movie[0],"year": movie[2],"index": i})
    return res


def get_movie(i):
    """Renvoie le dictionnaire détaillé du film à l'index i"""
    movie = movies[i]
    name = movie[0]
    actors_indexes = movie[1]
    year = movie[2]
    actors_list = []

    for j in actors_indexes:
        actor_name = actors[j][0]
        actors_list.append({"name": actor_name,"index": j})

    return {"name": name,"year": year,"actors": actors_list}



# Actors
def search_actor(name):
    """Renvoie une liste des acteurs dont le nom contient la chaîne donnée"""
    results = []
    name_lower = name.lower()

    for i in range(len(actors)):
        actor = actors[i]
        actor_name = actor[0]
        if name_lower in actor_name.lower():
            results.append({"name": actor_name,"index": i})

    return results


def get_actor(i):
    """Renvoie les détails de l'acteur à l'index i"""
    actor = actors[i]
    actor_name = actor[0]
    movies_indexes = actor[1]
    movies_list = []

    for j in movies_indexes:
        movie = movies[j]
        movie_name = movie[0]
        movie_year = movie[2]

        movies_list.append({"name": movie_name,"year": movie_year,"index": j})

    return {"name": actor_name,"movies": movies_list}


def get_costars(actor_id):
    """
    Renvoie une liste contenant les costars d'un acteur
    """
    actor_movies = get_actor(actor_id)['movies']
    unique_costars = {}

    for movie in actor_movies:
        costars = get_movie(movie['index'])['actors']

        for costar in costars:
            if costar['index'] != actor_id:
                unique_costars[costar['index']] = costar

    return list(unique_costars.values())


# Distance
def movie_path(origin, destination):
    """
    Calcule le chemin le plus court entre deux acteurs (BFS)
    Renvoie la distance et la liste des étapes
    """
    if origin == destination:
        return 0, [actors[origin][0]]

    visited = {origin: (None, None)}
    queue = deque([origin])

    while queue:
        current_actor = queue.popleft()

        for movie_id in actors[current_actor][1]:

            for next_actor in movies[movie_id][1]:

                if next_actor in visited:
                    continue

                visited[next_actor] = (current_actor, movie_id)

                if next_actor == destination:
                    return reconstruct_path(visited, destination)

                queue.append(next_actor)

    return -1, []


def reconstruct_path(visited, destination):
    """Reconstruit le chemin complet en remontant le dictionnaire visited"""

    path = []
    distance = 0
    current = destination

    while current is not None:
        actor_name = actors[current][0]
        path.append(actor_name)

        previous_actor, movie_id = visited[current]
        if movie_id is not None:
            movie_title = movies[movie_id][0]
            path.append(movie_title)
            distance += 1

        current = previous_actor

    path.reverse()
    return distance, path



# Fonctions supplémentaires pour le tri et la pagination
def sort_order(data_list, start, limit, order_key=None):
    """
    Trie une liste, puis la renvoie respectant la limite
    """
    safe_limit = min(limit, 100)

    if order_key:

        def sort_key(item):
            val = item.get(order_key, "")
            if val is not None and val != "":
                return val
            if order_key == 'year':
                return 0
            return ""

        try:
            data_list = sorted(data_list, key=sort_key)
        except (TypeError, KeyError):
            pass # sans order si le tri ne marche pas

    end = start + safe_limit
    return data_list[start:end]


def apply_movie_filters(results, filter_param):
    """
    Applique la chaine des filtres.
    Exemple : "year:1990,year:1991" retourne les films de 1990 OU 1991
    """
    if not filter_param:
        return results

    pairs = filter_param.split(',')

    requested_years = set()

    for pair in pairs:
        if ':' not in pair:
            continue
        key, value = pair.split(':', 1)

        if key == 'year':
            requested_years.add(int(value))

    if requested_years:
        results = [m for m in results if m.get('year') in requested_years]

    return results

Writing shared_code.py


In [48]:
%%writefile app_api.py
from flask import Blueprint, jsonify, request, abort
from werkzeug.exceptions import HTTPException
import shared_code

# Blueprint 1/2 : api
api_bp = Blueprint('api', __name__)


# Movies
@api_bp.route('/movies')
def list_movies_api():
    """Renvoie une liste des films en appliquant la fonction sort_order"""
    start = request.args.get('start', 0, type=int)
    limit = request.args.get('limit', 100, type=int)
    order = request.args.get('order')

    # Filtrer les films dont le titre est vide
    data = [{'name': m[0], 'year': m[2], 'index': i} for i, m in enumerate(shared_code.movies) if m[0] != ""]

    return jsonify(shared_code.sort_order(data, start, limit, order))

@api_bp.route('/movies/<int:movie_id>')
def get_movie_api(movie_id):
    """Renvoie les infos du film"""
    return jsonify(shared_code.get_movie(movie_id))


# Actors
@api_bp.route('/actors')
def list_actors_api():
    """Renvoie une liste des acteurs en appliquant la fonction sort_order"""
    start = request.args.get('start', 0, type=int)
    limit = request.args.get('limit', 100, type=int)
    order = request.args.get('order')

    data = [{'name': a[0], 'index': i} for i, a in enumerate(shared_code.actors) if a[0]]

    return jsonify(shared_code.sort_order(data, start, limit, order))

@api_bp.route('/actors/<int:actor_id>')
def get_actor_api(actor_id):
    """Renvoie les infos de l'acteur"""
    return jsonify(shared_code.get_actor(actor_id))

@api_bp.route('/actors/<int:actor_id>/costars')
def get_costars_api(actor_id):
    """Renvoie la liste des costars d'un acteur en appliquant la fonction sort_order"""
    costars = shared_code.get_costars(actor_id)
    return jsonify(shared_code.sort_order(costars, 0, 100))


# Search
@api_bp.route('/search/movies/<searchString>')
def search_movies_api(searchString):
    """Renvoie une liste des films correspondant à la recherche en appliquant la fonction sort_order (sans ordre)"""
    start = request.args.get('start', 0, type=int)
    limit = request.args.get('limit', 100, type=int)

    results = shared_code.search_movie(searchString)

    filter_param = request.args.get('filter')
    results = shared_code.apply_movie_filters(results, filter_param)

    return jsonify(shared_code.sort_order(results, start, limit))

@api_bp.route('/search/actors/<searchString>')
def search_actors_api(searchString):
    """Renvoie une liste des acteurs correspondant à la recherche en appliquant la fonction sort_order (sans ordre)"""
    start = request.args.get('start', 0, type=int)
    limit = request.args.get('limit', 100, type=int)

    results = shared_code.search_actor(searchString)

    return jsonify(shared_code.sort_order(results, start, limit))


# Distance
@api_bp.route('/actors/<int:id_orig>/distance/<int:id_dest>')
def get_distance_api(id_orig, id_dest):
    """Calcule le chemin le plus court entre deux acteurs"""
    return jsonify(shared_code.movie_path(id_orig, id_dest))


# Errors
@api_bp.errorhandler(ValueError)
@api_bp.errorhandler(TypeError)
def handle_bad_request(e):
    """Renvoie une erreur 400 lors d'une mauvaise requête (problème de type ou de valeur)"""
    return jsonify({"error": f"Bad Request: {str(e)}", "status": 400}), 400

@api_bp.errorhandler(IndexError)
def handle_not_found(e):
    """Renvoie une erreur 404 lorsque la ressource n'est pas trouvée"""
    return jsonify({"error": "Resource not found", "status": 404}), 404

Writing app_api.py


In [49]:
%%writefile run.py
from flask import Flask, jsonify, request, render_template
from werkzeug.exceptions import HTTPException
import gzip, json, shared_code
from app_api import api_bp
from app_site import site_bp
from pathlib import Path

app = Flask(__name__)
data_dir = Path.home() / "Downloads"

print("Server started")
with gzip.open(data_dir / 'movies.json.gz', 'rt') as f:
    shared_code.movies = json.load(f)
with gzip.open(data_dir / 'actors.json.gz', 'rt') as f:
    shared_code.actors = json.load(f)

print("movies and actors Loaded into memory")

app.register_blueprint(api_bp, url_prefix='/api')
app.register_blueprint(site_bp)

#print(shared_code.actors[:10])
#print()
#print(shared_code.movies[:10])


@app.errorhandler(Exception)
def handle_exception(e):
    """
    Gestionnaire global qui intercepte toutes les erreurs.
    Il décide de renvoyer du JSON ou du HTML en fonction du contexte de la requête.
    """

    if isinstance(e, HTTPException):
        code = e.code
        message = e.description
    else:
        code = 500
        message = str(e)

    if request.path.startswith('/api'):
        return jsonify({"error": message, "status": code}), code
    else:
        return render_template('error.html', title=f"Error {code}", description=message), code


if __name__ == "__main__":
    app.run(port=5000, debug=True, use_reloader=False)

Writing run.py


## Exercise 7. Test a Web API

Using `pytest`, write a program that checks that the API made in the previous exercise works as expected.

Your answer here.

In [50]:
%%writefile test_api.py
import requests
import pytest

BASE_URL = "http://127.0.0.1:5000/api"

# Movies
def test_get_movie_by_id():
    """Vérifie qu'un ID de film valide renvoie les bonnes clés"""
    response = requests.get(f"{BASE_URL}/movies/1")
    assert response.status_code == 200
    data = response.json()
    assert 'name' in data
    assert 'year' in data
    assert 'actors' in data
    assert isinstance(data['actors'], list)

def test_list_movies_pagination():
    """Teste si start et limit fonctionnent bien"""
    response = requests.get(f"{BASE_URL}/movies?start=10&limit=5")
    assert response.status_code == 200
    data = response.json()
    assert len(data) == 5

def test_list_movies_limit_cap():
    """Vérifie que l'API bloque bien à 100 résultats max"""
    response = requests.get(f"{BASE_URL}/movies?limit=200")
    assert response.status_code == 200
    assert len(response.json()) == 100

def test_list_movies_sorting():
    """Vérifie que le tri par année fonctionne dans l'ordre chrono"""
    response = requests.get(f"{BASE_URL}/movies?limit=20&order=year")
    assert response.status_code == 200
    data = response.json()
    years = []
    for m in data:
        years.append(m['year'])
    assert years == sorted(years)

# Actors
def test_get_actor_by_id():
    """Vérifie qu'un ID d'acteur valide renvoie les bonnes infos"""
    response = requests.get(f"{BASE_URL}/actors/1")
    assert response.status_code == 200
    data = response.json()
    assert 'name' in data
    assert 'movies' in data

def test_list_actors_sorting():
    """Teste le tri alphabétique des acteurs"""
    response = requests.get(f"{BASE_URL}/actors?limit=20&order=name")
    assert response.status_code == 200
    data = response.json()
    names = []
    for a in data:
        names.append(a['name'])
    assert names == sorted(names)

def test_get_costars():
    """Vérifie que la route des collègues renvoie bien une liste"""
    response = requests.get(f"{BASE_URL}/actors/1/costars")
    assert response.status_code == 200
    data = response.json()
    assert isinstance(data, list)
    if len(data) > 0:
        first_costar = data[0]
        assert 'name' in first_costar
        assert 'index' in first_costar

# Search Movies
def test_search_movies_multiple_filters_or_logic():
    """Teste si on récupère bien les films des deux années demandées"""
    response = requests.get(f"{BASE_URL}/search/movies/The?filter=year:1990,year:1991")
    assert response.status_code == 200
    data = response.json()
    assert len(data) > 0
    for movie in data:
        year = movie['year']
        assert year == 1990 or year == 1991

def test_search_movies_bad_year_value():
    """Vérifie qu'une année invalide provoque une erreur 400"""
    response = requests.get(f"{BASE_URL}/search/movies/Star?filter=year:abcd")
    assert response.status_code == 400
    data = response.json()
    assert "invalid literal" in data['error']

def test_search_movies_mixed_junk_filter():
    """Teste si l'API ignore les filtres mal écrits tout en gardant les bons"""
    response = requests.get(f"{BASE_URL}/search/movies/The?filter=year:1990,malformed")
    assert response.status_code == 200
    data = response.json()
    assert len(data) > 0
    for movie in data:
        assert movie['year'] == 1990

# Search Actors
def test_search_actors_case_insensitive():
    """Vérifie que la recherche marche peu importe les majuscules"""
    response = requests.get(f"{BASE_URL}/search/actors/blanche")
    assert response.status_code == 200
    data = response.json()
    found = False
    for actor in data:
        if "Blanche" in actor['name']:
            found = True
            break
    assert found == True

def test_actor_distance_identity():
    """La distance entre un acteur et lui-même doit être 0"""
    response = requests.get(f"{BASE_URL}/actors/0/distance/0")
    assert response.status_code == 200
    result = response.json()
    dist = result[0]
    path = result[1]
    assert dist == 0
    assert len(path) == 1

def test_actor_distance_invalid_ids():
    """Vérifie que des ID inconnus ne font pas planter l'API"""
    response = requests.get(f"{BASE_URL}/actors/0/distance/9999999")
    assert response.status_code == 200
    data = response.json()
    assert data[0] == -1
    assert data[1] == []

# Errors
def test_error_404_consistent_json():
    """S'assure que les erreurs 404 renvoient toujours le même format JSON"""
    response = requests.get(f"{BASE_URL}/movies/9999999")
    assert response.status_code == 404
    data = response.json()
    assert 'error' in data
    assert 'status' in data
    assert data["status"] == 404

def test_error_400_bad_filter_format():
    """Vérifie que le mauvais format de filtre renvoie une erreur 400"""
    response = requests.get(f"{BASE_URL}/search/movies/Star?filter=year:error")
    assert response.status_code == 400

Writing test_api.py


## Exercise 8. Make a Website that uses the Web API

Create a Python web server using Flask. Use the Web API you developed to offer the user a graphical Web interface. This interface allows the user to obtain, by entering relevant information into a Web form:

- The complete list of movies and the complete list of costars of an actor, possibly sorted alphabetically. This actor can be searched beforehand using a substring of characters appearing in her name.
- The colloration distance between two actors. As above, the actors can be searched beforehand using a substring of characters appearing in their names. Try to format a bit (not too much). For example:
  - The collaboration distance between Kevin Bacon and Jean Dujardin is 2.
  - Kevin bacon played in Wild things with Bill Murray;
  - Bill Murray played in The Monuments Men with Jean Dujardin.

Your answer here.

In [51]:
import os
os.makedirs('static', exist_ok=True)
os.makedirs('templates', exist_ok=True)

In [52]:
%%writefile app_site.py
from flask import Blueprint, render_template, request, abort, redirect, url_for
import requests

# Blueprint 2/2
site_bp = Blueprint('site', __name__)

# Pour faires les demandes à l'API
API_URL = "http://127.0.0.1:5000/api"


# Main
@site_bp.route('/')
def index():
    """Redirige vers la liste des films"""
    return redirect(url_for('site.list_movies'))

# Movies
@site_bp.route('/movies')
def list_movies():
    """
    Affiche la liste des films avec gestion de (start, limit) et (order)
    """
    start = get_int_param('start', 0)
    limit = get_int_param('limit', 100)
    order = request.args.get('order')

    api_params = {'start': start, 'limit': limit, 'order': order}
    results = get_from_api("movies", params=api_params) or []

    return render_template('list_movies.html', results=results, start=start, limit=limit, order=order)

@site_bp.route('/movies/<int:movie_id>')
def movie_details(movie_id):
    """Affiche la page d'un film"""
    movie = get_from_api(f"movies/{movie_id}")
    if not movie:
        abort(404, description="Movie not found.")
    return render_template('movie.html', movie=movie)

# Search Movies
@site_bp.route('/search/movies')
def search_movies_landing():
    """Affiche la page de recherche de films initiale (sans résultats)"""
    return render_template('search_movies.html', results=[], limit=100)

@site_bp.route('/search/movies/<searchString>')
def search_movies_site(searchString):
    """
    Effectue une recherche de films avec gestion des filtres (année) et applique le limit
    """
    year_filter = request.args.get('filter')

    start = get_int_param('start', 0)
    limit = get_int_param('limit', 100)

    api_params = {'start': start, 'limit': limit}
    if year_filter:
        api_params['filter'] = year_filter

    results = get_from_api(f"search/movies/{searchString}", params=api_params) or []

    return render_template('search_movies.html', results=results, query=searchString, limit=limit, start=start, year=year_filter)


# Actors
@site_bp.route('/actors')
def list_actors():
    """Affiche la liste des acteurs avec gestion (start, limit) et (order)"""
    start = get_int_param('start', 0)
    limit = get_int_param('limit', 100)
    order = request.args.get('order')

    api_params = {'start': start, 'limit': limit, 'order': order}
    results = get_from_api("actors", params=api_params) or []

    return render_template('list_actors.html', results=results, start=start, limit=limit, order=order)

@site_bp.route('/actors/<int:actor_id>')
def actor_details(actor_id):
    """Affiche la page d'un acteur"""
    actor = get_from_api(f"actors/{actor_id}")
    if not actor:
        abort(404, description="Actor not found.")

    costars = get_from_api(f"actors/{actor_id}/costars")
    if costars is None:
        costars = []

    return render_template('actor.html', actor=actor, actor_id=actor_id, costars=costars)


# Search Actors
@site_bp.route('/search/actors')
def search_actors_landing():
    """Affiche la page de recherche d'acteurs initiale (sans résultats)"""
    return render_template('search_actors.html', results=[], limit=100)

@site_bp.route('/search/actors/<searchString>')
def search_actors_site(searchString):
    """
    Effectue une recherche de acteurs avec application du (limit)
    """
    start = get_int_param('start', 0)
    limit = get_int_param('limit', 100)

    api_params = {'start': start, 'limit': limit}

    results = get_from_api(f"search/actors/{searchString}", params=api_params) or []

    return render_template('search_actors.html', results=results, query=searchString, limit=limit, start=start)

# Distance
@site_bp.route('/distance')
def distance_form():
    """Traite le formulaire de distance et redirige vers l'URL pour suivre le format demandé"""
    name1 = request.args.get('name1')
    name2 = request.args.get('name2')

    if name1 and name2:
        id1 = resolve_actor_id(name1)
        id2 = resolve_actor_id(name2)

        if id1 is not None and id2 is not None:
            return redirect(url_for('site.show_distance', id_orig=id1, id_dest=id2))
        else:
            return render_template('distance.html', error="Actor not found", name1=name1, name2=name2)

    return render_template('distance.html')


@site_bp.route('/actors/<int:id_orig>/distance/<int:id_dest>')
def show_distance(id_orig, id_dest):
    """Affiche le résultat du chemin le plus court entre deux acteurs"""

    # NOTE : L'accès direct à cette route contourne la validation faite dans /distance.
    # Cela déclenche directement l'algorithme de chemin (coûteux).
    # Nous avons choisi de ne pas dupliquer la logique de vérification ici par souci de simplicité.

    data = get_from_api(f"actors/{id_orig}/distance/{id_dest}")

    if data is None:
        abort(500)

    dist, path = data

    if path:
        # Si on a un chemin, On peut récupérer les noms directement
        name1 = path[0]
        name2 = path[-1]
    else:
        # Sinon On récupére les noms depuis l'API
        actor1 = get_from_api(f"actors/{id_orig}")
        name1 = actor1['name'] if actor1 else f"Unknown ID ({id_orig})"

        actor2 = get_from_api(f"actors/{id_dest}")
        name2 = actor2['name'] if actor2 else f"Unknown ID ({id_dest})"

    if dist == -1:
        error = "No connection found."
    else:
        error = None

    return render_template('distance.html', distance=dist, path=path, error=error, name1=name1, name2=name2)


# Fonctions Helpers
def get_from_api(endpoint, params=None):
    """
    Envoyer des requêtes HTTP GET à l'API
    """
    try:
        url = f"{API_URL}/{endpoint}"
        response = requests.get(url, params=params)
        if response.status_code == 404:
            return None

        # Vérifie les erreurs de l'API avant de lever une exception.
        # Si l'API renvoie un message d'erreur JSON clair, nous voulons l'afficher
        if not response.ok:
            try:
                # Tente de lire le champ "error" depuis la réponse JSON de l'API
                error_json = response.json()
                if 'error' in error_json:
                    # Utilise le message propre de l'API
                    abort(500, description=error_json['error'])
            except ValueError:
                # Si la réponse n'était pas du JSON, on ignore et on laisse raise_for_status gérer
                pass

        response.raise_for_status()

        return response.json()

    except requests.RequestException as e:
        # En cas d'échec de connexion (ex: serveur éteint)
        # on affiche un message générique
        abort(500, description="Could not connect to the API server.")

def resolve_actor_id(user_input):
    """
    Vérifie si l'entrée est un ID ou un nom et renvoie l'index de l'acteur
    On l’utilise pour permettre à l’utilisateur d’entrer le nom d’un acteur
    dans la page de distance, au lieu d’être obligé d’utiliser uniquement les index
    """
    if not user_input:
        return None

    if user_input.isdigit():
        actor_id = int(user_input)
        # Pour éviter d'appeller la fonction movie_path sur une id qui n'éxiste pas
        actor = get_from_api(f"actors/{actor_id}")
        if actor:
            return actor_id
        else:
            return None

    results = get_from_api(f"search/actors/{user_input}")

    if results:
        return results[0]['index']
    return None

def get_int_param(key, default):
    val = request.args.get(key, '')
    if val.isdigit():
        return int(val)
    return default


Writing app_site.py


In [53]:
%%writefile static/style.css
/* Gruvbox Dark Palette */
:root {
    --bg0: #282828;
    --bg1: #3c3836;
    --bg2: #504945;
    --fg: #ebdbb2;
    --red: #fb4934;
    --green: #b8bb26;
    --yellow: #fabd2f;
    --blue: #83a598;
    --purple: #d3869b;
    --aqua: #8ec07c;
    --gray: #a89984;
    --orange: #fe8019;
}

body {
    font-family: 'Consolas', 'Monaco', 'Courier New', monospace;
    line-height: 1.6;
    margin: 0;
    padding: 20px;
    background-color: var(--bg0);
    color: var(--fg);
}

.container {
    max-width: 1100px;
    margin: 0 auto;
    background-color: var(--bg1);
    padding: 40px;
    border-radius: 8px;
    box-shadow: 0 4px 15px rgba(0, 0, 0, 0.5);
    border: 1px solid var(--bg2);
}

nav {
    display: flex;
    justify-content: center; /* Center the links */
    flex-wrap: wrap;         /* Wrap only on very small screens */
    gap: 20px;               /* Consistent spacing between items */
    margin-bottom: 30px;
    border-bottom: 2px solid var(--bg2);
    padding-bottom: 15px;
    align-items: center;
}

nav a {
    color: var(--yellow);
    text-decoration: none;
    font-weight: bold;
    font-size: 1em;
}

nav a:hover {
    color: var(--orange);
    text-decoration: underline;
}

/* Headers */
h1, h2, h3 { color: var(--green); margin-top: 0; }
h1 { border-bottom: 1px solid var(--bg2); padding-bottom: 10px; }

/* SEARCH BAR CONTAINER */
.search-bar {
    background-color: var(--bg2);
    padding: 15px;
    border-radius: 8px;
    margin-bottom: 20px;
    border: 1px solid var(--bg1);
    display: flex;
    gap: 10px;
    align-items: center;
    flex-wrap: wrap;
}

.search-bar form {
    display: flex;
    width: 100%;
    gap: 10px;
}

/* INPUTS & SELECTS */
input[type="text"], input[type="number"], select {
    background-color: var(--bg0);
    border: 1px solid var(--gray);
    padding: 10px;
    border-radius: 4px;
    color: var(--fg);
    font-family: inherit;
    font-size: 1em;
    height: 42px; /* Fixed height for alignment */
    box-sizing: border-box;
}

input[type="text"] { flex: 2; }
input[type="number"] { flex: 1; }
select { flex: 1; cursor: pointer; }

input:focus, select:focus {
    outline: none;
    border-color: var(--blue);
}

/* HIDE SPINNER (Up/Down Arrows) on Number Inputs */
input[type=number]::-webkit-inner-spin-button,
input[type=number]::-webkit-outer-spin-button {
  -webkit-appearance: none;
  margin: 0;
}
input[type=number] {
  -moz-appearance: textfield; /* Firefox */
}

/* BUTTONS */
button {
    height: 42px; /* Match input height */
    padding: 0 24px;
    background-color: var(--blue);
    color: var(--bg0);
    border: none;
    border-radius: 4px;
    cursor: pointer;
    font-weight: bold;
    font-family: inherit;
    font-size: 1em;
    transition: background 0.2s, transform 0.1s;
    white-space: nowrap;
}

button:hover {
    background-color: var(--aqua);
}

button:active {
    transform: translateY(1px);
}

button:disabled {
    background-color: var(--bg2);
    color: var(--gray);
    cursor: not-allowed;
}

/* Lists & Cards */
ul { padding: 0; }

li, .movie-card {
    background-color: var(--bg0);
    margin: 10px 0;
    padding: 15px;
    border-radius: 4px;
    list-style: none;
    border: 1px solid var(--bg2);
    transition: border-color 0.2s;
}

li:hover, .movie-card:hover {
    border-color: var(--blue);
}

/* Links */
a { color: var(--blue); text-decoration: none; }
a:hover { color: var(--purple); text-decoration: underline; }

.link-item, .item-link {
    color: var(--blue);
    font-weight: bold;
    font-size: 1.1em;
}

.year-tag {
    color: var(--gray);
    font-size: 0.9em;
    margin-left: 8px;
}

/* Distance Path Styling */
.path-box {
    background: var(--bg0);
    padding: 20px;
    border-radius: 8px;
    border: 1px solid var(--bg2);
    display: flex;
    flex-direction: column;
    align-items: center;
}
.path-node {
    font-weight: bold;
    color: var(--yellow);
    font-size: 1.2rem;
    padding: 5px 10px;
    background: var(--bg1);
    border-radius: 4px;
}
.path-arrow { color: var(--orange); font-weight: bold;}
.path-step { margin: 10px 0; color: var(--gray); font-style: italic; }

/* Pagination Container */
.pagination {
    display: flex;
    justify-content: space-between;
    margin-top: 30px;
    padding-top: 20px;
    border-top: 1px solid var(--bg2);
}

/* Sort Bar (New Class) */
.sort-bar {
    display: flex;
    justify-content: flex-end;
    margin-bottom: 10px;
    align-items: center;
    gap: 10px;
}

Writing static/style.css


In [54]:
%%writefile templates/base.html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>IMDB Explorer</title>
    <link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>
<body>
    <div class="container">
        <nav style="margin-bottom: 20px; border-bottom: 1px solid #eee; padding-bottom: 10px;">
            <a href="{{ url_for('site.list_movies') }}" style="margin-right: 15px;">List Movies</a>
            <a href="{{ url_for('site.search_movies_landing') }}" style="margin-right: 15px;">Search Movies</a>
            <span style="color: #504945; margin-right: 15px;">|</span>
            <a href="{{ url_for('site.list_actors') }}" style="margin-right: 15px;">List Actors</a>
            <a href="{{ url_for('site.search_actors_landing') }}" style="margin-right: 15px;">Search Actors</a>
            <span style="color: #504945; margin-right: 15px;">|</span>
            <a href="{{ url_for('site.distance_form') }}">Distance Calculator</a>
        </nav>

        {% block content %}{% endblock %}
    </div>
</body>
</html>

Writing templates/base.html


In [55]:
%%writefile templates/list_movies.html
{% extends 'base.html' %}

{% block content %}
    <div style="margin-bottom: 20px;">
        <h3>List Movies</h3>
        <form action="{{ url_for('site.list_movies') }}" method="GET" style="display: flex; gap: 10px; align-items: center; flex-wrap: wrap;">

            <input type="number" name="start" placeholder="Start"
                   value="{{ start if start != 0 else '' }}"
                   min="0" style="width: 70px;">

            <input type="number" name="limit" placeholder="Limit"
                   value="{{ limit if limit != 100 else '' }}"
                   min="1" max="100" style="width: 70px;">

            <select name="order" style="width: 100px;">
                <option value="">Index</option>
                <option value="name" {% if order == 'name' %}selected{% endif %}>Name</option>
                <option value="year" {% if order == 'year' %}selected{% endif %}>Year</option>
            </select>

            <button type="submit" style="background-color: var(--gray);">List</button>
        </form>
    </div>

    <hr style="border-color: var(--bg2); margin: 20px 0;">

    <h3>Full Movie List</h3>

    {% if results %}
        {% for movie in results %}
            <div class="movie-card">
                <a href="{{ url_for('site.movie_details', movie_id=movie.index) }}">
                    <strong>{{ movie.name }}</strong>
                </a>
                <span class="movie-year">({{ movie.year }})</span>
                <span class="year-tag" style="font-size: 0.8em;">ID: {{ movie.index }}</span>
            </div>
        {% endfor %}

        <div class="pagination">
            {% if start is defined and start > 0 %}
                <a href="{{ url_for('site.list_movies', order=order, start=start - limit, limit=limit) }}">
                    <button>← Previous</button>
                </a>
            {% else %}
                <button disabled>← Previous</button>
            {% endif %}

            {% if results|length == limit %}
                <a href="{{ url_for('site.list_movies', order=order, start=(start or 0) + limit, limit=limit) }}">
                    <button>Next →</button>
                </a>
            {% else %}
                 <button disabled>Next →</button>
            {% endif %}
        </div>
    {% else %}
        <p>No movies found.</p>
    {% endif %}
{% endblock %}

Writing templates/list_movies.html


In [56]:
%%writefile templates/search_movies.html
{% extends 'base.html' %}

{% block content %}
    <div style="margin-bottom: 20px;">
        <h3>Search Movies</h3>
        <form id="movieSearchForm" onsubmit="handleMovieSearch(event)" style="display: flex; gap: 10px; flex-wrap: wrap;">

            <input type="text" id="qInput" placeholder="Title..." value="{{ query or '' }}" required style="flex: 2; min-width: 120px;">
            <input type="text" id="yearInput" placeholder="Year(s)" value="{{ year or '' }}" style="flex: 1; min-width: 80px;">

            <input type="number" id="limitSearchInput" placeholder="Limit"
                   value="{{ limit if limit != 100 else '' }}"
                   min="1" max="100" style="width: 70px;">

            <button type="submit">Search</button>
        </form>
    </div>

    <hr style="border-color: var(--bg2); margin: 20px 0;">

    {% if query %}
        <h3>Search Results for: <span style="color:var(--yellow)">"{{ query }}"</span></h3>
    {% endif %}

    {% if results %}
        {% for movie in results %}
            <div class="movie-card">
                <a href="{{ url_for('site.movie_details', movie_id=movie.index) }}">
                    <strong>{{ movie.name }}</strong>
                </a>
                <span class="movie-year">({{ movie.year }})</span>
                <span class="year-tag" style="font-size: 0.8em;">ID: {{ movie.index }}</span>
            </div>
        {% endfor %}

        <div class="pagination">
            {% if start|default(0) > 0 %}
                <a href="{{ url_for('site.search_movies_site', searchString=query, start=start|default(0) - limit, limit=limit, filter=year) }}">
                    <button>← Previous</button>
                </a>
            {% else %}
                <button disabled>← Previous</button>
            {% endif %}

            {% if results|length == limit %}
                <a href="{{ url_for('site.search_movies_site', searchString=query, start=start|default(0) + limit, limit=limit, filter=year) }}">
                    <button>Next →</button>
                </a>
            {% else %}
                 <button disabled>Next →</button>
            {% endif %}
        </div>

    {% elif query %}
        <p>No movies found matching your criteria.</p>
    {% endif %}

    <script>
        function handleMovieSearch(event) {
            event.preventDefault();
            const q = document.getElementById('qInput').value.trim();
            let limit = document.getElementById('limitSearchInput').value;
            if (!limit) limit = 100;
            const year = document.getElementById('yearInput').value.trim();

            if (!q) return;

            let url = "/search/movies/" + encodeURIComponent(q) + "?limit=" + limit;

            if (year) {
                const years = year.split(',').map(y => y.trim()).filter(y => !isNaN(y) && y !== "");
                if (years.length > 0) {
                     const filterStr = years.map(y => "year:" + y).join(",");
                     url += "&filter=" + encodeURIComponent(filterStr);
                }
            }
            window.location.href = url;
        }
    </script>
{% endblock %}

Writing templates/search_movies.html


In [57]:
%%writefile templates/movie.html
{% extends 'base.html' %}

{% block content %}
<nav>
    <a href="{{ url_for('site.index') }}">← Back</a>
</nav>

<header>
    <h1>{{ movie.name }}</h1>
    <p class="year-tag">Release Year: {{ movie.year }}</p>
</header>

<h3 style="margin-top: 30px;">Cast Members</h3>
<ul>
    {% if movie.actors %}
        {% for actor in movie.actors %}
            <li>
                <a class="link-item" href="{{ url_for('site.actor_details', actor_id=actor.index) }}">
                    {{ actor.name }}
                </a>
                <span class="year-tag">(ID: {{ actor.index }})</span>
            </li>
        {% endfor %}
    {% else %}
        <li>No cast information found.</li>
    {% endif %}
</ul>
{% endblock %}

Writing templates/movie.html


In [58]:
%%writefile templates/list_actors.html
{% extends 'base.html' %}

{% block content %}
    <div style="margin-bottom: 20px;">
        <h3>List Actors</h3>
        <form action="{{ url_for('site.list_actors') }}" method="GET" style="display: flex; gap: 10px; align-items: center; flex-wrap: wrap;">

            <input type="number" name="start" placeholder="Start"
                   value="{{ start if start != 0 else '' }}"
                   min="0" style="width: 70px;">

            <input type="number" name="limit" placeholder="Limit"
                   value="{{ limit if limit != 100 else '' }}"
                   min="1" max="100" style="width: 70px;">

            <select name="order" style="width: 100px;">
                <option value="">Index</option>
                <option value="name" {% if order == 'name' %}selected{% endif %}>Name</option>
            </select>

            <button type="submit" style="background-color: var(--gray);">List</button>
        </form>
    </div>

    <hr style="border-color: var(--bg2); margin: 20px 0;">

    <h3>Full Actor List</h3>

    {% if results %}
        <ul>
        {% for actor in results %}
            <li>
                <a class="link-item" href="{{ url_for('site.actor_details', actor_id=actor.index) }}">
                    {{ actor.name }}
                </a>
                <span class="year-tag">(ID: {{ actor.index }})</span>
            </li>
        {% endfor %}
        </ul>

        <div class="pagination">
            {% if start is defined and start > 0 %}
                <a href="{{ url_for('site.list_actors', order=order, start=start - limit, limit=limit) }}">
                    <button>← Previous</button>
                </a>
            {% else %}
                <button disabled>← Previous</button>
            {% endif %}

            {% if results|length == limit %}
                <a href="{{ url_for('site.list_actors', order=order, start=(start or 0) + limit, limit=limit) }}">
                    <button>Next →</button>
                </a>
            {% else %}
                 <button disabled>Next →</button>
            {% endif %}
        </div>
    {% else %}
        <p>No actors found.</p>
    {% endif %}
{% endblock %}

Writing templates/list_actors.html


In [59]:
%%writefile templates/search_actors.html
{% extends 'base.html' %}

{% block content %}
    <div style="margin-bottom: 20px;">
        <h3>Search Actors</h3>
        <form id="actorSearchForm" onsubmit="handleActorSearch(event)" style="display: flex; gap: 10px; flex-wrap: wrap;">

            <input type="text" id="qInput" placeholder="Actor Name..." value="{{ query or '' }}" required style="flex: 2; min-width: 150px;">

            <input type="number" id="limitSearchInput" placeholder="Limit"
                   value="{{ limit if limit != 100 else '' }}"
                   min="1" max="100" style="width: 70px;">

            <button type="submit">Search</button>
        </form>
    </div>

    <hr style="border-color: var(--bg2); margin: 20px 0;">

    {% if query %}
        <h3>Search Results for: <span style="color:var(--yellow)">"{{ query }}"</span></h3>
    {% endif %}

    {% if results %}
        <ul>
        {% for actor in results %}
            <li>
                <a class="link-item" href="{{ url_for('site.actor_details', actor_id=actor.index) }}">
                    {{ actor.name }}
                </a>
                <span class="year-tag">(ID: {{ actor.index }})</span>
            </li>
        {% endfor %}
        </ul>

        <div class="pagination">
            {% if start is defined and start > 0 %}
                <a href="{{ url_for('site.search_actors_site', searchString=query, start=start - limit, limit=limit) }}">
                    <button>← Previous</button>
                </a>
            {% else %}
                <button disabled>← Previous</button>
            {% endif %}

            {% if results|length == limit %}
                <a href="{{ url_for('site.search_actors_site', searchString=query, start=start + limit, limit=limit) }}">
                    <button>Next →</button>
                </a>
            {% else %}
                 <button disabled>Next →</button>
            {% endif %}
        </div>

    {% elif query %}
        <p>No actors found.</p>
    {% endif %}

    <script>
        function handleActorSearch(event) {
            event.preventDefault();
            const q = document.getElementById('qInput').value.trim();
            let limit = document.getElementById('limitSearchInput').value;
            if (!limit) limit = 100;

            if (!q) return;

            let url = "/search/actors/" + encodeURIComponent(q) + "?limit=" + limit;
            window.location.href = url;
        }
    </script>
{% endblock %}

Writing templates/search_actors.html


In [60]:
%%writefile templates/actor.html
{% extends 'base.html' %}

{% block content %}
<nav>
    <a href="{{ url_for('site.list_actors') }}">← Back</a>
</nav>

<header>
    <h1 style="color: #556b2f;">{{ actor.name }}</h1>
    <p class="year-tag">Actor Profile (ID: {{ actor_id }})</p>
</header>

<div style="display: flex; gap: 40px; margin-top: 30px; flex-wrap: wrap; align-items: flex-start;">

    <div style="flex: 1; min-width: 300px;">
        <h3 style="color: #8da399; border-bottom: 2px solid #eee; padding-bottom: 10px;">Filmography</h3>
        <ul>
            {% if actor.movies %}
                {% for movie in actor.movies %}
                    <li>
                        <a class="link-item" href="{{ url_for('site.movie_details', movie_id=movie.index) }}">
                            {{ movie.name }}
                        </a>
                        <span class="year-tag">({{ movie.year }})</span>
                        <span class="year-tag" style="font-size: 0.8em;">ID: {{ movie.index }}</span>
                    </li>
                {% endfor %}
            {% else %}
                <li>No movies found.</li>
            {% endif %}
        </ul>
    </div>

    <div style="flex: 1; min-width: 300px;">
        <h3 style="color: #8da399; border-bottom: 2px solid #eee; padding-bottom: 10px;">Co-stars</h3>

        {% if costars %}
            <ul>
                {% for star in costars %}
                    <li>
                        <a class="link-item" href="{{ url_for('site.actor_details', actor_id=star.index) }}">
                            {{ star.name }}
                        </a>
                        <span class="year-tag">(ID: {{ star.index }})</span>
                    </li>
                {% endfor %}
            </ul>
        {% else %}
            <p>No co-stars found.</p>
        {% endif %}
    </div>

</div>
{% endblock %}

Writing templates/actor.html


In [61]:
%%writefile templates/distance.html
{% extends 'base.html' %}

{% block content %}
    <h2>Find Actor Connection</h2>
    <p>Find the shortest path between two actors by entering their <strong>Names</strong> or <strong>IDs</strong>.</p>

    <form action="{{ url_for('site.distance_form') }}" method="GET" style="display: flex; gap: 10px; align-items: center;">
        <input type="text" name="name1" placeholder="Actor 1 (e.g. Tom Hanks)" value="{{ name1 or '' }}" required style="flex: 1;">
        <input type="text" name="name2" placeholder="Actor 2 (e.g. Kevin Bacon)" value="{{ name2 or '' }}" required style="flex: 1;">
        <button type="submit">Find Path</button>
    </form>

    <hr style="border-color: #504945; margin: 30px 0;">

    {% if path %}
        <div style="margin-bottom: 20px; padding: 15px; background: var(--bg2); border-radius: 4px;">
            <strong>Results for:</strong> {{ name1 }} ↔ {{ name2 }} <br>
            <strong>Distance:</strong> {{ distance }} degree(s)
        </div>

        <div class="path-box">
            {% for item in path %}
                {% if loop.index0 % 2 == 0 %}
                    <span class="path-node">{{ item }}</span>
                {% else %}
                    <div class="path-step">
                        <span class="path-arrow">↓</span>
                        via <em>"{{ item }}"</em>
                        <span class="path-arrow">↓</span>
                    </div>
                {% endif %}
            {% endfor %}
        </div>
    {% elif error %}
        <div style="color: var(--bg0); background: var(--red); padding: 15px; border-radius: 4px; font-weight: bold;">
            {{ error }}
        </div>
    {% endif %}
{% endblock %}

Writing templates/distance.html


In [62]:
%%writefile templates/error.html
{% extends 'base.html' %}

{% block content %}
    <div style="text-align: center; padding: 60px 0;">
        <h1 style="color: var(--red); font-size: 4em; margin-bottom: 10px; border: none;">
            {{ title }}
        </h1>

        <div style="width: 50px; height: 4px; background: var(--yellow); margin: 0 auto 30px auto;"></div>

        <p style="font-size: 1.4em; color: var(--fg); margin-bottom: 40px; font-weight: bold;">
            {{ description }}
        </p>

        <a href="{{ url_for('site.index') }}" style="text-decoration: none;">
            <button style="font-size: 1.2em; padding: 15px 30px; background-color: var(--blue); color: var(--bg0); border-radius: 4px; border: none; cursor: pointer;">
                ← Return to Home
            </button>
        </a>
    </div>
{% endblock %}

Writing templates/error.html


In [63]:
# Après l'exécution des cellules,
# on peut lancer le serveur.
# Pour voir les sorties standard du serveur,
# il est possible de le lancer dans un terminal séparé.

# Méthode proposée par l'IA, censée fonctionner sur toutes les plateformes (espérons-le)

import subprocess
import sys

with open("server.log", "w") as f:
    process = subprocess.Popen(
        [sys.executable, "run.py"],
        stdout=f,
        stderr=f
    )

# Non bloquant : le notebook continue son exécution
print("Script launched in background! Logs go to server.log")
print("Visit : http://127.0.0.1:5000")

Script launched in background! Logs go to server.log
Visit : http://127.0.0.1:5000


In [64]:
!pytest test_api.py

============================= test session starts ==============================
platform linux -- Python 3.13.5, pytest-9.0.2, pluggy-1.6.0
rootdir: Removed
plugins: anyio-4.11.0
collected 15 items                                                             

test_api.py ...............                                              [100%]

============================== 15 passed in 6.79s ==============================
